# Nifty 50 Swing Pipeline — Colab GPU Training

Trains the XGBoost swing-prediction pipeline on Colab's GPU instead of your laptop.

**Before running:** set the runtime to GPU — `Runtime ▸ Change runtime type ▸ Hardware accelerator: GPU`.

Then run the cells top to bottom.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU, then rerun'

## 2. Clone the repo

If the repo is **private**, replace the URL with `https://<TOKEN>@github.com/...` using a GitHub personal access token.

In [ ]:
import os

REPO_URL = 'https://github.com/aditya33agrawal/XGBoost-Swing-Trading-Prediction-System.git'
BRANCH   = 'main'  # change if you push to a different branch
REPO_DIR = '/content/swing'

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

# The pipeline package lives at the repo root (run_pipeline.py + src/).
%cd $REPO_DIR

## 3. Install dependencies

Colab already ships CUDA-enabled `xgboost`, plus `numpy/pandas/scipy/scikit-learn/pyarrow`. The pipeline needs **`optuna`**, **`yfinance`**, and **`supabase`** on top of that.

`supabase` is required *only if* you want DB writes (next section). It's a pure HTTP client, so it's safe to install — it doesn't touch numpy. Without it, the pipeline silently stays JSON-only even when credentials are set.

**Deliberately NOT installed:**
- `pandas-ta` — its old build is incompatible with Colab's numpy 2.x / Python 3.12 and corrupts the numpy install (the `cannot import name '_center'` error). `src/features/engineer.py` has built-in pure-pandas fallbacks for every indicator, so dropping it changes nothing.
- `shap`, `mlflow`, `duckdb` — not imported at runtime (storage degrades gracefully).

If you already hit the numpy error in this session, do **Runtime ▸ Restart session** once, then run from cell 1 again.

In [ ]:
# Minimal install — does NOT touch numpy, so the kernel stays healthy.
# `supabase` is a pure HTTP client (httpx/postgrest), safe alongside numpy 2.x.
# It's required for DB writes; without it the pipeline stays JSON-only even
# when SUPABASE_URL / SUPABASE_SECRET_KEY are set.
!pip install -q optuna yfinance supabase

import numpy, xgboost as xgb
print('numpy  ', numpy.__version__)
print('xgboost', xgb.__version__)

## 3b. (Optional) Connect Supabase

By default the pipeline runs in **JSON-only** mode and writes everything to
`outputs/` — that's the `Supabase not configured` message, and it's fine for
training. If you *also* want predictions/runs written to Supabase, add
`SUPABASE_URL` and `SUPABASE_SECRET_KEY` to **Colab Secrets** (🔑 in the left
sidebar) and run the next cell. Skip it to stay JSON-only.

In [ ]:
# (Optional) Load Supabase credentials from Colab Secrets so the pipeline writes
# to the DB instead of JSON-only. The `!python run_pipeline.py` subprocess below
# inherits these env vars. Skip this cell to stay in JSON-only mode.
import os

def _load_secret(*names):
    from google.colab import userdata
    for name in names:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            continue
    return None

url = _load_secret('SUPABASE_URL')
key = _load_secret('SUPABASE_SECRET_KEY', 'SUPABASE_KEY')  # secret key = DB write access
if url and key:
    os.environ['SUPABASE_URL'] = url
    os.environ['SUPABASE_SECRET_KEY'] = key
    print('Supabase credentials loaded — pipeline will write to Supabase + outputs/')
else:
    print('No Supabase secrets found — staying JSON-only (writes to outputs/).')
    print('Add SUPABASE_URL + SUPABASE_SECRET_KEY in Colab Secrets (🔑) to enable.')

# Verify the package is importable AND a live connection can be made, so the
# "Supabase not configured" message in the training logs can't surprise you.
if url and key:
    try:
        from supabase import create_client
    except ImportError:
        print('⚠️  supabase package NOT installed — rerun the install cell (cell 3). '
              'Without it the pipeline silently falls back to JSON-only.')
    else:
        try:
            create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SECRET_KEY'])
            print('✅ Supabase client connected — DB writes will work.')
        except Exception as exc:
            print(f'⚠️  Supabase connection failed: {exc}')

## 4. Train on the GPU

`--device auto` detects the GPU automatically; the logs will say `device=cuda`.
Set `LOG_LEVEL=DEBUG` for even more detail. Lower `--trials` for a quick test run.

In [ ]:
import os
os.environ['LOG_LEVEL'] = 'INFO'   # set to 'DEBUG' for more

!python run_pipeline.py --trials 50 --device auto --log-level INFO

# save_bundle failures are non-fatal in the pipeline (logged as a WARNING and
# easy to miss in a long Optuna log), so check explicitly here.  The most common
# cause of an EMPTY registry/ is a stale clone: the registry phase lives in
# src/pipeline/runner.py, so if it isn't in the pushed commit on `main` the run
# produces predictions but no bundle.  Re-run cell 4 (git pull) if that happens.
import glob
if not glob.glob('/content/swing/registry/bundles/model_*'):
    print("⚠️  No bundle under registry/bundles/. Check the log for a "
          "'Bundle save failed' WARNING. If there is none, the clone is stale — "
          "re-run cell 4 (git pull) so runner.py includes the registry phase, then rerun.")
else:
    print('✅ Model bundle written to registry/bundles/')
    !ls -la /content/swing/registry/bundles/


## 5. (Optional) Save the trained artifacts to Google Drive

Colab wipes `/content` when the runtime disconnects. Copy `models/`, `registry/`, and `data/` to Drive to keep them.

Each run is saved under a **date-stamped folder** so you keep a history:

```
swing_outputs/
└── 2026-06-21/
    ├── models/      ← best_params.json
    ├── registry/    ← trained booster bundle + manifest
    └── data/        ← cached prices (duckdb + parquet)
```

In [ ]:
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')

# Date-stamped destination so every run is preserved: swing_outputs/<YYYY-MM-DD>/
RUN_DATE = datetime.now().strftime('%Y-%m-%d')
DEST     = f'/content/drive/MyDrive/swing_outputs/{RUN_DATE}'
import os, shutil
os.makedirs(DEST, exist_ok=True)

# Copy every artifact folder the run produces, only if it exists — and say so if
# not, instead of silently swallowing the cp error (the old '2>/dev/null || true'
# made a missing registry/ look identical to a successful copy).
#   models/    → best_params.json
#   registry/  → trained booster bundle + manifest + metrics (the deployable model)
#   reports/   → drift report (HTML)
#   outputs/   → signals JSON/CSV, predictions journal, model_runs, portfolio
#   data/      → cached prices (duckdb + parquet)
for name in ('models', 'registry', 'reports', 'outputs', 'data'):
    src = f'/content/swing/{name}'
    if os.path.exists(src):
        shutil.copytree(src, f'{DEST}/{name}', dirs_exist_ok=True)
        print(f'✅ copied {name}/')
    else:
        msg = f'⚠️  {name}/ does not exist in /content/swing — nothing to copy '
        msg += '(check the training cell output above for why it was never created)'
        print(msg)

print(f'\nSaved to Drive ▸ swing_outputs/{RUN_DATE}')
!ls -R "$DEST" | head -60
